# 00 — Notebook exploratório · Despesas de Viamão/RS (2019–2024)

**Fase 4** do roadmap ([`TAREFAS.md`](../TAREFAS.md)) · Princípio **P6**.

Objetivo: um **sanity check** dos dados já processados antes de qualquer análise. Este notebook **não** filtra, corrige ou interpreta — só descreve o que existe no consolidado e confere os totais contra os Excels de auditoria gerados pelo ETL (`etl/02_processa.py`).

Fonte dos dados: **TCE-RS / SIAPC** (dados de responsabilidade da Prefeitura de Viamão, enviados via SIAPC e **não auditados** pelo Tribunal). Cada linha do consolidado é **uma operação** e `tipo_operacao` é sempre **E**mpenho, **L**iquidação ou **P**agamento.

Roteiro obrigatório (P6):
1. Contagem de linhas por ano
2. Tipos de operação (E/L/P) e quantidades
3. Funções e órgãos únicos
4. Top 10 credores brutos (antes de qualquer filtro)
5. Verificação de nulos nas colunas críticas
6. Totais de liquidação e pagamento por ano (conferência com a auditoria)

## Setup — imports e caminhos

Primeira célula: imports e constantes (P5). Caminhos relativos à raiz do projeto (P5: nunca caminho absoluto do dev).

In [1]:
from pathlib import Path

import pandas as pd

# Raiz do projeto = pasta-mãe de notebooks/ (o notebook roda a partir de notebooks/).
RAIZ_PROJETO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ARQ_PARQUET = RAIZ_PROJETO / "dados" / "processed" / "viamao_despesa_consolidado.parquet"
DIR_AUDITORIA = RAIZ_PROJETO / "dados" / "auditoria"

# tipo_operacao assume exatamente E/L/P nos 6 anos (verificado no ETL).
MAPA_TIPO = {"E": "Empenho", "L": "Liquidação", "P": "Pagamento"}

# Exibição de valores em R$ com separador de milhar, 2 casas (só formatação de tela).
pd.options.display.float_format = lambda v: f"{v:,.2f}"

if not ARQ_PARQUET.exists():
    raise FileNotFoundError(
        f"Parquet não encontrado: {ARQ_PARQUET}. "
        "Rode antes: python etl/02_processa.py (P4 — não pular etapas)."
    )
print("Raiz do projeto :", RAIZ_PROJETO)
print("Parquet         :", ARQ_PARQUET.name)

Raiz do projeto : C:\Users\vitor\OneDrive\Meu_curso_Economia\setor_publico
Parquet         : viamao_despesa_consolidado.parquet


## Carga do consolidado

Lê o único insumo desta fase — o parquet consolidado (regenerável pelo ETL). Inspeção visual com `shape`, `dtypes` e `.head()`.

In [2]:
df = pd.read_parquet(ARQ_PARQUET)
print("Dimensões:", df.shape[0], "linhas ×", df.shape[1], "colunas")
df.dtypes

Dimensões: 408953 linhas × 31 colunas


ano                                   int64
ano_recebimento                       Int64
mes_recebimento                       Int64
ano_empenho                           Int64
ano_operacao                          Int64
tipo_operacao                           str
tipo_operacao_desc                      str
cd_orgao_orcamentario                   str
nome_orgao_orcamentario                 str
cd_unidade_orcamentaria                 str
nome_unidade_orcamentaria               str
cd_funcao                               str
ds_funcao                               str
cd_subfuncao                            str
ds_subfuncao                            str
cd_elemento                             str
cd_rubrica                              str
ds_rubrica                              str
cd_credor                               str
nm_credor                               str
tp_pessoa                               str
cnpj_cpf                                str
nr_empenho                      

In [3]:
# Amostra das colunas mais usadas nas análises 3.1/3.2/3.3.
colunas_amostra = [
    "ano", "tipo_operacao", "tipo_operacao_desc", "nome_orgao_orcamentario",
    "ds_funcao", "nm_credor", "cnpj_cpf", "vl_liquidacao", "vl_pagamento",
]
df[colunas_amostra].head()

,ano,tipo_operacao,tipo_operacao_desc,nome_orgao_orcamentario,ds_funcao,nm_credor,cnpj_cpf,vl_liquidacao,vl_pagamento
0,2019,E,Empenho,SEC MUN DA ADMINISTRACAO,Previdência Social,ANTAO RODRIGUES SILVA,NaN,NaN,NaN
1,2019,L,Liquidação,SEC MUN DA ADMINISTRACAO,Previdência Social,ANTAO RODRIGUES SILVA,NaN,200.23,NaN
2,2019,E,Empenho,SEC MUN DA ADMINISTRACAO,Administração,VILMAR FERNANDES LIMA,47697644053,NaN,NaN
3,2019,L,Liquidação,SEC MUN DA ADMINISTRACAO,Administração,VILMAR FERNANDES LIMA,47697644053,216.42,NaN
4,2019,E,Empenho,SEC MUN DA FAZENDA,Administração,LAURA MORAIS LEAL,1796664014,NaN,NaN


## 1. Contagem de linhas por ano

Quantas operações há em cada arquivo anual. Espera-se crescimento gradual (mais operações a cada ano) e nenhum ano faltando (2019–2024 completos).

In [4]:
linhas_por_ano = df["ano"].value_counts().sort_index()
linhas_por_ano.index.name = "ano"
resumo_linhas = linhas_por_ano.rename("linhas").to_frame()
resumo_linhas.loc["TOTAL"] = resumo_linhas["linhas"].sum()
print("Anos presentes:", sorted(df['ano'].unique()))
resumo_linhas

Anos presentes: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]


,linhas
ano,
2019,59715
2020,62411
2021,65224
2022,68968
2023,75901
2024,76734
TOTAL,408953


## 2. Tipos de operação (E/L/P) e quantidades

Confere que só existem os três códigos esperados — Empenho, Liquidação e Pagamento — e a quantidade de cada um. As análises usam **Liquidação** (3.1 e 3.2) e **Pagamento** (3.3).

In [5]:
tipos = df["tipo_operacao"].value_counts()
tab_tipos = tipos.rename("quantidade").to_frame()
tab_tipos["descricao"] = tab_tipos.index.map(MAPA_TIPO)
tab_tipos["% do total"] = (tab_tipos["quantidade"] / len(df) * 100).round(1)

# P1: garante que não há código fora de E/L/P (nada inventado a jusante).
inesperados = set(df["tipo_operacao"].unique()) - set(MAPA_TIPO)
assert not inesperados, f"tipo_operacao inesperado: {inesperados}"
print("Só há os tipos esperados (E/L/P):", sorted(df['tipo_operacao'].unique()))
tab_tipos[["descricao", "quantidade", "% do total"]]

Só há os tipos esperados (E/L/P): ['E', 'L', 'P']


,descricao,quantidade,% do total
tipo_operacao,,,
P,Pagamento,194150,47.50
L,Liquidação,120745,29.50
E,Empenho,94058,23.00


In [6]:
# Cruzamento tipo × ano — visão de como cada arquivo se distribui entre E/L/P.
pd.crosstab(df["ano"], df["tipo_operacao_desc"], margins=True, margins_name="TOTAL")

tipo_operacao_desc,Empenho,Liquidação,Pagamento,TOTAL
ano,,,,
2019,13675,18693,27347,59715
2020,13568,19198,29645,62411
2021,14950,19300,30974,65224
2022,16115,19797,33056,68968
2023,18023,21689,36189,75901
2024,17727,22068,36939,76734
TOTAL,94058,120745,194150,408953


## 3. Funções e órgãos únicos

O universo de rótulos que estruturam as seções 3.2 (funções, Portaria 42/1999) e 3.1 (órgãos/classificação institucional). Aqui só listamos — sem agrupar valores ainda.

In [7]:
orgaos = df["nome_orgao_orcamentario"].dropna().unique()
orgaos = sorted(orgaos)
print(f"Órgãos orçamentários únicos: {len(orgaos)}\n")
for o in orgaos:
    print("  •", o)

Órgãos orçamentários únicos: 25

  • GABINETE DO PREFEITO
  • PROCURADORIA GERAL
  • PROCURADORIA GERAL DO MUNICIPIO
  • SEC DE GESTAO E REL. INSTITUCIONAIS
  • SEC GERAL DE GOVERNO
  • SEC MUN DA ADMINISTRACAO
  • SEC MUN DA CIDADANIA E ASSIST SOCIAL
  • SEC MUN DA EDUCACAO
  • SEC MUN DA FAZENDA
  • SEC MUN DA SAUDE
  • SEC MUN DE AGRICULTURA E ABASTEC
  • SEC MUN DE CIDADANIA E ASSIST SOCIAL
  • SEC MUN DE CULTURA
  • SEC MUN DE DESENV ECON INDUST COM
  • SEC MUN DE DESENV ECON INDUST COM E TUR
  • SEC MUN DE ESPORTE E LAZER
  • SEC MUN DE MEIO AMBIENTE
  • SEC MUN DE OBRAS E SERV PUBLICOS
  • SEC MUN DE OBRAS E SERVICOS
  • SEC MUN DE PLANEJ URBAN E HABITACAO
  • SEC MUN DE PLANEJ, URBANISMO E HABITACAO
  • SEC MUN DE TRANSPORTE E TRANSITO
  • SEC MUN DE TRANSPORTES E MANUT DA FROTA
  • SECRETARIA MUNICIPAL DA EDUCACAO
  • SECRETARIA MUNICIPAL DE TURISMO


In [8]:
funcoes = df["ds_funcao"].dropna().unique()
funcoes = sorted(funcoes)
print(f"Funções (ds_funcao) únicas: {len(funcoes)}\n")
for f in funcoes:
    print("  •", f)

Funções (ds_funcao) únicas: 36

  • ADMINISTRACAO
  • AGRICULTURA
  • ASSISTENCIA SOCIAL
  • Administração
  • Agricultura
  • Assistência Social
  • COMERCIO E SERVICOS
  • CULTURA
  • Comércio e Serviços
  • Cultura
  • DESPORTO E LAZER
  • DIREITOS DA CIDADANIA
  • Desporto e Lazer
  • Direitos da Cidadania
  • EDUCACAO
  • ENCARGOS ESPECIAIS
  • Educação
  • Encargos Especiais
  • GESTAO AMBIENTAL
  • Gestão Ambiental
  • HABITACAO
  • Habitação
  • PREVIDENCIA SOCIAL
  • Previdência Social
  • SANEAMENTO
  • SAUDE
  • SEGURANCA PUBLICA
  • Saneamento
  • Saúde
  • Segurança Pública
  • TRABALHO
  • TRANSPORTE
  • Trabalho
  • Transporte
  • URBANISMO
  • Urbanismo


In [9]:
# Quantas funções/órgãos distintos aparecem em cada ano (checagem de estabilidade do schema).
df.groupby("ano").agg(
    orgaos_distintos=("nome_orgao_orcamentario", "nunique"),
    funcoes_distintas=("ds_funcao", "nunique"),
)

,orgaos_distintos,funcoes_distintas
ano,,
2019,22,18
2020,22,17
2021,22,16
2022,22,16
2023,18,13
2024,18,15


## 4. Top 10 credores brutos (antes de qualquer filtro)

Ranking **bruto**: agrupa por `nm_credor` e soma `vl_pagamento` sobre **todas** as linhas, sem nenhum tratamento. É só uma fotografia — o tratamento correto (separar pagadores internos como *FOLHA DE PAGAMENTO* e *IPREV* dos fornecedores externos, e recortar o ano de 2024) é feito na **Seção 3.3**. Por isso aparecem aqui credores internos, esperadamente no topo.

In [10]:
total_pago = df["vl_pagamento"].sum()
top10_credores = (
    df.groupby("nm_credor")["vl_pagamento"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .rename("vl_pagamento_total")
    .to_frame()
)
top10_credores["% do total pago"] = (top10_credores["vl_pagamento_total"] / total_pago * 100).round(1)
print(f"Total pago (bruto, 2019–2024): R$ {total_pago:,.2f}")
print(f"Credores distintos: {df['nm_credor'].nunique()}")
top10_credores

Total pago (bruto, 2019–2024): R$ 3,620,867,070.21
Credores distintos: 7022


,vl_pagamento_total,% do total pago
nm_credor,,
FOLHA DE PAGAMENTO,"1,300,155,893.04",35.90
IPREV - INST DE PREVIDENCIA DOS SERV. DE VIAMAO,"251,669,753.74",7.00
URBAN SERVICOS E TRANSPORTES LTDA,"144,630,747.26",4.00
INSTITUTO SOCIO-EDUCACIONAL DA BIODIVERSIDADE - IN,"102,996,110.89",2.80
DCS-CL CONSTRUTORA E PAVIMENTADORA LTDA,"90,580,682.92",2.50
GREEN CARD S/A REFEICOES COMERCIO E SERVICOS,"70,399,530.49",1.90
ORGANIZACAO DA SOCIEDADE CIVIL IN SAUDE - INSTITUT,"67,557,155.64",1.90
INSTITUTO DOS LAGOS - RIO,"62,863,240.74",1.70
SODEXO PASS DO BRASIL SERVICOS E COMERCIO S.A,"60,023,816.55",1.70


## 5. Verificação de nulos nas colunas críticas

Atenção ao **desenho da fonte**: como cada linha é *uma* operação, as colunas de valor são nulas fora do seu tipo — `vl_liquidacao` só é preenchida em linhas **L**, e `vl_pagamento` só em linhas **P**. Portanto o nulo relevante é o que aparece *dentro* do filtro certo, não no dataset inteiro.

In [11]:
colunas_criticas = [
    "ano", "tipo_operacao", "nome_orgao_orcamentario", "ds_funcao",
    "nm_credor", "cnpj_cpf", "vl_liquidacao", "vl_pagamento",
]
nulos = df[colunas_criticas].isna().sum().rename("nulos").to_frame()
nulos["% de nulos"] = (nulos["nulos"] / len(df) * 100).round(1)
nulos

,nulos,% de nulos
ano,0,0.00
tipo_operacao,0,0.00
nome_orgao_orcamentario,0,0.00
ds_funcao,0,0.00
nm_credor,0,0.00
cnpj_cpf,123809,30.30
vl_liquidacao,288208,70.50
vl_pagamento,214803,52.50


In [12]:
# Nulos ESTRUTURAIS (esperados): valor nulo fora do tipo de operação correspondente.
# Se o desenho está consistente, nº de vl_liquidacao preenchidos == nº de linhas L,
# e nº de vl_pagamento preenchidos == nº de linhas P.
n_L = int((df["tipo_operacao"] == "L").sum())
n_P = int((df["tipo_operacao"] == "P").sum())
vl_liq_preenchidos = int(df["vl_liquidacao"].notna().sum())
vl_pag_preenchidos = int(df["vl_pagamento"].notna().sum())
print(f"linhas L = {n_L:>7} | vl_liquidacao preenchidos = {vl_liq_preenchidos:>7}")
print(f"linhas P = {n_P:>7} | vl_pagamento  preenchidos = {vl_pag_preenchidos:>7}")
assert vl_liq_preenchidos == n_L, "vl_liquidacao preenchido fora de linhas L!"
assert vl_pag_preenchidos == n_P, "vl_pagamento preenchido fora de linhas P!"
print("\nOK: cada valor só aparece no seu tipo de operação (nulos de valor são estruturais).")

linhas L =  120745 | vl_liquidacao preenchidos =  120745
linhas P =  194150 | vl_pagamento  preenchidos =  194150

OK: cada valor só aparece no seu tipo de operação (nulos de valor são estruturais).


In [13]:
# Nulos REAIS dentro do filtro certo — os que poderiam prejudicar as análises.
liq = df[df["tipo_operacao"] == "L"]
pag = df[df["tipo_operacao"] == "P"]
checagens = {
    "L: vl_liquidacao nulo": int(liq["vl_liquidacao"].isna().sum()),
    "L: nome_orgao nulo": int(liq["nome_orgao_orcamentario"].isna().sum()),
    "L: ds_funcao nulo": int(liq["ds_funcao"].isna().sum()),
    "P: vl_pagamento nulo": int(pag["vl_pagamento"].isna().sum()),
    "P: nm_credor nulo": int(pag["nm_credor"].isna().sum()),
    "P: cnpj_cpf nulo": int(pag["cnpj_cpf"].isna().sum()),
}
reais = pd.Series(checagens, name="registros nulos").to_frame()
reais["% no filtro"] = [
    round(v / len(liq) * 100, 1) if k.startswith("L") else round(v / len(pag) * 100, 1)
    for k, v in checagens.items()
]
print("Observação (documentada no ETL): cnpj_cpf ausente em ~30% — a fonte não")
print("guarda zeros à esquerda; nm_credor, ao contrário, está 100% preenchido.")
reais

Observação (documentada no ETL): cnpj_cpf ausente em ~30% — a fonte não
guarda zeros à esquerda; nm_credor, ao contrário, está 100% preenchido.


,registros nulos,% no filtro
L: vl_liquidacao nulo,0,0.00
L: nome_orgao nulo,0,0.00
L: ds_funcao nulo,0,0.00
P: vl_pagamento nulo,0,0.00
P: nm_credor nulo,0,0.00
P: cnpj_cpf nulo,70884,36.50


## 6. Totais de liquidação e pagamento por ano (conferência com a auditoria)

O sanity check final e mais importante (P2/P4): os totais recalculados a partir do parquet têm de **bater exatamente** com os Excels de auditoria `dados/auditoria/viamao_despesa_{ANO}_auditoria.xlsx` (aba *Metadados*), que são a evidência primária do trabalho. Valores **nominais** (sem deflação — a deflação pelo IPCA é feita nas seções 3.1/3.2).

In [14]:
# Totais recalculados do parquet: soma LÍQUIDA (estornos negativos mantidos — P1).
tot_liq = df[df["tipo_operacao"] == "L"].groupby("ano")["vl_liquidacao"].sum()
tot_pag = df[df["tipo_operacao"] == "P"].groupby("ano")["vl_pagamento"].sum()
totais_parquet = pd.DataFrame({"liquidacao_R$": tot_liq, "pagamento_R$": tot_pag})
totais_parquet.loc["TOTAL"] = totais_parquet.sum()
totais_parquet

,liquidacao_R$,pagamento_R$
ano,,
2019,"489,488,993.67","441,581,461.90"
2020,"554,619,153.77","526,196,573.05"
2021,"620,202,152.14","590,205,606.57"
2022,"683,783,421.80","665,733,559.66"
2023,"675,417,284.70","662,235,805.44"
2024,"739,475,339.09","734,914,063.59"
TOTAL,"3,762,986,345.17","3,620,867,070.21"


In [15]:
# Lê a soma registrada em cada Excel de auditoria (aba Metadados) e compara.
# Os Excels são regeneráveis (python etl/02_processa.py) e gitignorados; se algum
# não estiver presente localmente, a célula avisa e segue (não quebra — P4).
ROT_LIQ = "Soma vl_liquidacao (líquida, R$)"
ROT_PAG = "Soma vl_pagamento (líquida, R$)"

linhas_conf = []
for ano in range(2019, 2025):
    arq = DIR_AUDITORIA / f"viamao_despesa_{ano}_auditoria.xlsx"
    parquet_liq = float(tot_liq.get(ano, float("nan")))
    parquet_pag = float(tot_pag.get(ano, float("nan")))
    if not arq.exists():
        print(f"[AVISO] auditoria de {ano} ausente ({arq.name}) — regenere com etl/02_processa.py")
        continue
    meta = pd.read_excel(arq, sheet_name="Metadados").set_index("campo")["valor"]
    aud_liq = float(meta[ROT_LIQ])
    aud_pag = float(meta[ROT_PAG])
    linhas_conf.append({
        "ano": ano,
        "liq_parquet": parquet_liq, "liq_auditoria": aud_liq,
        "pag_parquet": parquet_pag, "pag_auditoria": aud_pag,
        "liq_bate": abs(parquet_liq - aud_liq) < 0.01,
        "pag_bate": abs(parquet_pag - aud_pag) < 0.01,
    })

conferencia = pd.DataFrame(linhas_conf).set_index("ano")
conferencia

,liq_parquet,liq_auditoria,pag_parquet,pag_auditoria,liq_bate,pag_bate
ano,,,,,,
2019,"489,488,993.67","489,488,993.67","441,581,461.90","441,581,461.90",True,True
2020,"554,619,153.77","554,619,153.77","526,196,573.05","526,196,573.05",True,True
2021,"620,202,152.14","620,202,152.14","590,205,606.57","590,205,606.57",True,True
2022,"683,783,421.80","683,783,421.80","665,733,559.66","665,733,559.66",True,True
2023,"675,417,284.70","675,417,284.70","662,235,805.44","662,235,805.44",True,True
2024,"739,475,339.09","739,475,339.09","734,914,063.59","734,914,063.59",True,True


In [16]:
# Veredito: todos os anos disponíveis têm de bater (diferença < R$ 0,01).
if conferencia.empty:
    print("Nenhum Excel de auditoria encontrado — rode etl/02_processa.py e reexecute.")
else:
    todos_batem = bool(conferencia[["liq_bate", "pag_bate"]].all().all())
    assert todos_batem, "Divergência entre parquet e auditoria — investigar o ETL!"
    print(f"OK — {len(conferencia)} anos conferidos; liquidação e pagamento batem com a auditoria.")

OK — 6 anos conferidos; liquidação e pagamento batem com a auditoria.


## Conclusão do sanity check

- 6 anos presentes (2019–2024), sem lacunas; nº de operações cresce a cada ano.
- Só existem os tipos **E / L / P**; nenhum código inesperado.
- Órgãos e funções listados e estáveis entre anos — base para 3.1 e 3.2.
- Top 10 credores brutos levantado; pagadores internos identificados para tratamento na 3.3.
- Nulos das colunas de valor são **estruturais** (cada valor só no seu tipo de operação); único vazio material é `cnpj_cpf` (~30%, documentado no ETL).
- Totais de liquidação e pagamento por ano **conferem** com os Excels de auditoria (diferença < R$ 0,01).

Dados íntegros e compreendidos. Próxima fase: **3.1 · Institucional** (`notebooks/31_institucional.ipynb`).

---
Fonte dos dados: **TCE-RS/SIAPC**. Elaboração própria.